In [1]:
import pandas as pd
import gc
import time
df = pd.read_csv('../ex04/fines.csv')
print("Размер загруженных данных:", df.shape)

Размер загруженных данных: (930, 6)


перебор с помощью цикла + iloc+ append

In [4]:
def calculate_loop(df):
    results = []
    for i in range(len(df)):
        row = df.iloc[i]
        if row['Refund'] != 0: 
            result = row['Fines'] / row['Refund'] * row['Year']
        else:
            result = 0
        results.append(result)
    return results
start_time = time.time()
df['calculated_loop'] = calculate_loop(df)
loop_time = time.time() - start_time
print(f"Время loop: {loop_time:.4f} секунд")

Время loop: 0.1168 секунд


 перебор с помощью iterrows()

In [6]:
def calculate_iterrows(df):
    results = []
    for index, row in df.iterrows():
        if row['Refund'] != 0:
            result = row['Fines'] / row['Refund'] * row['Year']
        else:
            result = 0
        results.append(result)
    return results
start_time = time.time()
df['calculated_iterrows'] = calculate_iterrows(df)
iterrows_time = time.time() - start_time
print(f"время для iterrows: {iterrows_time:.4f} секунд")

время для iterrows: 0.1402 секунд


In [8]:
def calculate_apply(df):
    return df.apply(lambda row: row['Fines'] / row['Refund'] * row['Year'] 
                   if row['Refund'] != 0 else 0, axis=1)
start_time = time.time()
df['calculated_apply'] = calculate_apply(df)
apply_time = time.time() - start_time
print(f"apply время: {apply_time:.4f} секунд")

apply время: 0.1382 секунд


векторизированные операции с series

In [10]:
def calculate_vectorized(df):
    return df['Fines'] / df['Refund'] * df['Year']

start_time = time.time()
df['calculated_vectorized'] = calculate_vectorized(df)
df['calculated_vectorized'] = df['calculated_vectorized'].fillna(0)
vectorized_time = time.time() - start_time
print(f"время vectorized: {vectorized_time:.4f} секунд")

время vectorized: 0.0029 секунд


векторизированные операции с values

In [12]:
def calculate_values(df):
    result = []
    fines = df['Fines'].values
    refunds = df['Refund'].values
    years = df['Year'].values
    
    for i in range(len(fines)):
        if refunds[i] != 0:
            result.append(fines[i] / refunds[i] * years[i])
        else:
            result.append(0)
    return result

start_time = time.time()
df['calculated_values'] = calculate_values(df)
values_time = time.time() - start_time
print(f"время values: {values_time:.4f} секунд")

время values: 0.0034 секунд


замеряем время доступа к строке

In [14]:
start_time = time.time()
row = df[df['CarNumber'] == 'O136HO197RUS']
no_index_time = time.time() - start_time
print(f"Поиск без индекса: {no_index_time:.6f} секунд")

Поиск без индекса: 0.005892 секунд


устанавливаем индекс по CarNumber

In [17]:
df_indexed = df.set_index('CarNumber')

In [21]:
start_time = time.time()
row = df_indexed.loc['O136HO197RUS']
with_index_time = time.time() - start_time
print(f"Поиск c индексом: {with_index_time:.6f} секунд")

Поиск c индексом: 0.000694 секунд


анализ использования памяти до оптимизации

In [24]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CarNumber              930 non-null    object 
 1   Refund                 930 non-null    int64  
 2   Fines                  930 non-null    float64
 3   Make                   930 non-null    object 
 4   Model                  918 non-null    object 
 5   Year                   930 non-null    int64  
 6   calculated_loop        930 non-null    float64
 7   calculated_iterrows    930 non-null    float64
 8   calculated_apply       930 non-null    float64
 9   calculated_vectorized  930 non-null    float64
 10  calculated_values      930 non-null    float64
dtypes: float64(6), int64(2), object(3)
memory usage: 211.1 KB


Создаем оптимизированную копию датафрейма, оптимизируем типы данных, добавляем 'strange' с оптимизированным типом

In [26]:
optimized_df = df.copy()
optimized_df['CarNumber'] = optimized_df['CarNumber'].astype('category')
optimized_df['Make'] = optimized_df['Make'].astype('category')
optimized_df['Model'] = optimized_df['Model'].astype('category')
optimized_df['Refund'] = optimized_df['Refund'].astype('int8')
optimized_df['Fines'] = optimized_df['Fines'].astype('float32')
optimized_df['Year'] = optimized_df['Year'].astype('int16')

optimized_df['strange'] = (optimized_df['Fines'] / optimized_df['Refund'] * optimized_df['Year']).fillna(0).astype('float32')

анализ использования памяти после оптимизации

In [28]:
optimized_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   CarNumber              930 non-null    category
 1   Refund                 930 non-null    int8    
 2   Fines                  930 non-null    float32 
 3   Make                   930 non-null    category
 4   Model                  918 non-null    category
 5   Year                   930 non-null    int16   
 6   calculated_loop        930 non-null    float64 
 7   calculated_iterrows    930 non-null    float64 
 8   calculated_apply       930 non-null    float64 
 9   calculated_vectorized  930 non-null    float64 
 10  calculated_values      930 non-null    float64 
 11  strange                930 non-null    float32 
dtypes: category(3), float32(2), float64(5), int16(1), int8(1)
memory usage: 99.8 KB


очищаем память и удаляем исходный датафрейм

In [31]:
del df
gc.collect()

0